# SEFD-Plus: Reproducibility Notebook

**Paper:** SEFD-Plus: Uncertainty-Aware Fraud Detection with Human-in-the-Loop Governance  
**Conference:** IEEE CCECE 2026  
**Paper ID:** 2692212270  
**Author:** Haifaa Owayed (howay035@uottawa.ca)  
**Institution:** University of Ottawa

---

## 📋 Overview

This notebook reproduces the experimental results from the SEFD-Plus paper submitted to IEEE CCECE 2026.

**Key Results (from submitted paper):**
- **False Positive Reduction:** 19.3% (17,818 → 13,041)
- **Baseline FPR:** 10.42%
- **SEFD-Plus FPR:** 8.41%
- **Baseline TPR:** 79.05%
- **SEFD-Plus TPR:** 81.16%
- **F2 Score:** 0.5157 → 0.5701
- **Gray Zone:** 9.3% of transactions (16,476)
- **Statistical Significance:** p < 10⁻⁸⁵ (Fisher's exact test)
- **Annual Cost Savings:** $958,540 (34.4% reduction)

---

## 🎯 Experimental Setup

**Dataset:** IEEE-CIS Fraud Detection  
- Total transactions: 590,540
- Test set: 177,162 transactions
- Fraud rate: 3.5%

**Model:** XGBoost Ensemble (5 models with different random seeds)  
**Uncertainty Quantification:** Ensemble variance  
**Gray Zone Policy:** Fixed thresholds (θ_low = 0.05, θ_high = 0.95)

---

## ⏱️ Expected Runtime

- **Data Loading:** ~2 minutes
- **Model Training:** ~45 minutes (5 models × 9 minutes each)
- **Evaluation:** ~5 minutes
- **Total:** ~1 hour

---

## 📦 Required Files

Download the IEEE-CIS Fraud Detection dataset from Kaggle:
- `train_transaction.csv` (must contain `isFraud` column)

Place the file in the same directory as this notebook or upload it when prompted.

---

# 1. Setup and Imports

In [ ]:
# Install required packages (uncomment if running for the first time)
# !pip install xgboost scikit-learn scipy statsmodels pandas numpy

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, 
    roc_auc_score, 
    fbeta_score,
    precision_recall_curve,
    roc_curve
)
from scipy.stats import fisher_exact
import json
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✅ Packages imported successfully!")
print(f"XGBoost version: {xgb.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

---

# 2. Configuration

These parameters match the submitted paper exactly.

In [ ]:
# =============================================================================
# CONFIGURATION (matches submitted paper)
# =============================================================================

# Dataset configuration
TEST_SIZE = 0.3  # 30% for testing (177,162 transactions)

# XGBoost hyperparameters (optimized via grid search)
XGBOOST_PARAMS = {
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'gamma': 0,
    'reg_alpha': 0,
    'reg_lambda': 1,
    'scale_pos_weight': 1,
    'tree_method': 'hist',
    'device': 'cpu'  # Change to 'cuda' if GPU is available
}

# Ensemble configuration
N_MODELS = 5  # Number of models in ensemble
ENSEMBLE_SEEDS = [42, 123, 456, 789, 2024]  # Random seeds for diversity

# Gray Zone thresholds (from paper)
THETA_LOW = 0.05   # Lower threshold for SAFE zone
THETA_HIGH = 0.95  # Upper threshold for FLAGGED zone

# Baseline threshold (standard practice)
BASELINE_THRESHOLD = 0.5

# Cost parameters (from paper's cost-benefit analysis)
COST_FALSE_POSITIVE = 25    # Cost of blocking legitimate transaction
COST_FALSE_NEGATIVE = 100   # Cost of missing fraud
COST_HUMAN_REVIEW = 5       # Cost of manual review

# Bootstrap parameters for confidence intervals
N_BOOTSTRAP = 1000
CONFIDENCE_LEVEL = 0.95

# Output directory
OUTPUT_DIR = Path('../results')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print("✅ Configuration loaded")
print(f"\n📊 Key Parameters:")
print(f"  - Ensemble size: {N_MODELS} models")
print(f"  - Gray Zone thresholds: [{THETA_LOW}, {THETA_HIGH}]")
print(f"  - Baseline threshold: {BASELINE_THRESHOLD}")
print(f"  - Test size: {TEST_SIZE*100:.0f}%")

---

# 3. Data Loading and Preprocessing

Load the IEEE-CIS Fraud Detection dataset and prepare it for training.

In [ ]:
# =============================================================================
# DATA LOADING
# =============================================================================

# For Google Colab: Upload file
try:
    from google.colab import files
    print("📤 Please upload train_transaction.csv...")
    uploaded = files.upload()
    DATA_PATH = 'train_transaction.csv'
except ImportError:
    # For local execution: Specify file path
    DATA_PATH = '../data/train_transaction.csv'
    print(f"📂 Loading data from: {DATA_PATH}")

# Load dataset
print("\n⏳ Loading dataset...")
df = pd.read_csv(DATA_PATH)

print(f"✅ Dataset loaded successfully!")
print(f"\n📊 Dataset Statistics:")
print(f"  - Total transactions: {len(df):,}")
print(f"  - Features: {df.shape[1]}")
print(f"  - Fraud cases: {df['isFraud'].sum():,} ({df['isFraud'].mean()*100:.2f}%)")
print(f"  - Legitimate cases: {(~df['isFraud'].astype(bool)).sum():,} ({(1-df['isFraud'].mean())*100:.2f}%)")

In [ ]:
# =============================================================================
# DATA PREPROCESSING
# =============================================================================

print("\n⏳ Preprocessing data...")

# Separate features and target
X = df.drop('isFraud', axis=1)
y = df['isFraud']

# Handle categorical variables
categorical_cols = X.select_dtypes(include=['object']).columns
print(f"  - Categorical columns: {len(categorical_cols)}")

# Convert categorical to numeric (label encoding)
for col in categorical_cols:
    X[col] = pd.Categorical(X[col]).codes

# Handle missing values (fill with -999 to distinguish from 0)
X = X.fillna(-999)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_SEED,
    stratify=y  # Maintain fraud ratio in both sets
)

print(f"\n✅ Preprocessing complete!")
print(f"\n📊 Train/Test Split:")
print(f"  - Training set: {len(X_train):,} transactions ({len(X_train)/len(df)*100:.1f}%)")
print(f"  - Test set: {len(X_test):,} transactions ({len(X_test)/len(df)*100:.1f}%)")
print(f"  - Train fraud rate: {y_train.mean()*100:.2f}%")
print(f"  - Test fraud rate: {y_test.mean()*100:.2f}%")

---

# 4. Baseline Model Training

Train a single XGBoost model as the baseline (standard practice without uncertainty quantification).

In [ ]:
# =============================================================================
# BASELINE MODEL (Single XGBoost)
# =============================================================================

print("\n" + "="*80)
print("TRAINING BASELINE MODEL")
print("="*80)

# Train baseline model
print("\n⏳ Training baseline XGBoost model...")
baseline_model = xgb.XGBClassifier(**XGBOOST_PARAMS, random_state=RANDOM_SEED)
baseline_model.fit(X_train, y_train, verbose=False)

# Get predictions
baseline_probs = baseline_model.predict_proba(X_test)[:, 1]
baseline_preds = (baseline_probs >= BASELINE_THRESHOLD).astype(int)

# Calculate metrics
tn, fp, fn, tp = confusion_matrix(y_test, baseline_preds).ravel()
baseline_fpr = fp / (fp + tn)
baseline_tpr = tp / (tp + fn)
baseline_f2 = fbeta_score(y_test, baseline_preds, beta=2)
baseline_auc = roc_auc_score(y_test, baseline_probs)

print(f"\n✅ Baseline model trained!")
print(f"\n📊 Baseline Performance:")
print(f"  - AUC: {baseline_auc:.4f}")
print(f"  - TPR (Recall): {baseline_tpr*100:.2f}%")
print(f"  - FPR: {baseline_fpr*100:.2f}%")
print(f"  - F2 Score: {baseline_f2:.4f}")
print(f"\n📊 Confusion Matrix:")
print(f"  - True Positives (TP): {tp:,}")
print(f"  - False Positives (FP): {fp:,}")
print(f"  - True Negatives (TN): {tn:,}")
print(f"  - False Negatives (FN): {fn:,}")

# Store baseline results
baseline_results = {
    'confusion_matrix': {'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn)},
    'tpr': float(baseline_tpr),
    'fpr': float(baseline_fpr),
    'f2': float(baseline_f2),
    'auc': float(baseline_auc),
    'threshold': BASELINE_THRESHOLD
}

---

# 5. SEFD-Plus Ensemble Training

Train an ensemble of 5 XGBoost models with different random seeds to enable uncertainty quantification.

In [ ]:
# =============================================================================
# SEFD-PLUS ENSEMBLE TRAINING
# =============================================================================

print("\n" + "="*80)
print("TRAINING SEFD-PLUS ENSEMBLE")
print("="*80)

ensemble_models = []
ensemble_predictions = []

for i, seed in enumerate(ENSEMBLE_SEEDS, 1):
    print(f"\n⏳ Training model {i}/{N_MODELS} (seed={seed})...")
    
    # Train model with different seed
    model = xgb.XGBClassifier(**XGBOOST_PARAMS, random_state=seed)
    model.fit(X_train, y_train, verbose=False)
    
    # Get predictions
    probs = model.predict_proba(X_test)[:, 1]
    
    # Store model and predictions
    ensemble_models.append(model)
    ensemble_predictions.append(probs)
    
    print(f"  ✅ Model {i} trained (AUC: {roc_auc_score(y_test, probs):.4f})")

# Convert to numpy array for easier computation
ensemble_predictions = np.array(ensemble_predictions)

print(f"\n✅ All {N_MODELS} models trained successfully!")
print(f"\n📊 Ensemble Statistics:")
print(f"  - Mean AUC: {np.mean([roc_auc_score(y_test, pred) for pred in ensemble_predictions]):.4f}")
print(f"  - Std AUC: {np.std([roc_auc_score(y_test, pred) for pred in ensemble_predictions]):.4f}")

---

# 6. Uncertainty Quantification

Calculate uncertainty using ensemble variance (epistemic uncertainty).

In [ ]:
# =============================================================================
# UNCERTAINTY QUANTIFICATION
# =============================================================================

print("\n" + "="*80)
print("UNCERTAINTY QUANTIFICATION")
print("="*80)

# Calculate mean prediction (consensus)
mean_probs = np.mean(ensemble_predictions, axis=0)

# Calculate uncertainty (variance across ensemble)
uncertainty = np.var(ensemble_predictions, axis=0)

print(f"\n✅ Uncertainty calculated!")
print(f"\n📊 Uncertainty Statistics:")
print(f"  - Mean uncertainty: {np.mean(uncertainty):.6f}")
print(f"  - Median uncertainty: {np.median(uncertainty):.6f}")
print(f"  - Max uncertainty: {np.max(uncertainty):.6f}")
print(f"  - Min uncertainty: {np.min(uncertainty):.6f}")

# Analyze uncertainty distribution
print(f"\n📊 Uncertainty Percentiles:")
for percentile in [25, 50, 75, 90, 95, 99]:
    value = np.percentile(uncertainty, percentile)
    print(f"  - {percentile}th percentile: {value:.6f}")

---

# 7. Gray Zone Classification

Apply the Gray Zone policy to route uncertain transactions to human review.

In [ ]:
# =============================================================================
# GRAY ZONE CLASSIFICATION
# =============================================================================

print("\n" + "="*80)
print("GRAY ZONE CLASSIFICATION")
print("="*80)

# Apply Gray Zone thresholds
sefdplus_decisions = np.zeros(len(mean_probs), dtype=int)

# SAFE zone: p(x) < θ_low → Automatic approval (decision = 0)
safe_mask = mean_probs < THETA_LOW
sefdplus_decisions[safe_mask] = 0

# FLAGGED zone: p(x) > θ_high → Automatic rejection (decision = 1)
flagged_mask = mean_probs > THETA_HIGH
sefdplus_decisions[flagged_mask] = 1

# GRAY zone: θ_low ≤ p(x) ≤ θ_high → Human review (decision = -1)
gray_mask = ~safe_mask & ~flagged_mask
sefdplus_decisions[gray_mask] = -1

# Count transactions in each zone
n_safe = np.sum(safe_mask)
n_flagged = np.sum(flagged_mask)
n_gray = np.sum(gray_mask)

print(f"\n✅ Gray Zone classification complete!")
print(f"\n📊 Zone Distribution:")
print(f"  - SAFE zone: {n_safe:,} ({n_safe/len(mean_probs)*100:.2f}%)")
print(f"  - GRAY zone: {n_gray:,} ({n_gray/len(mean_probs)*100:.2f}%)")
print(f"  - FLAGGED zone: {n_flagged:,} ({n_flagged/len(mean_probs)*100:.2f}%)")

# Analyze fraud distribution in each zone
print(f"\n📊 Fraud Distribution by Zone:")
print(f"  - SAFE zone fraud rate: {y_test[safe_mask].mean()*100:.2f}%")
print(f"  - GRAY zone fraud rate: {y_test[gray_mask].mean()*100:.2f}%")
print(f"  - FLAGGED zone fraud rate: {y_test[flagged_mask].mean()*100:.2f}%")

# Calculate Gray Zone enrichment
overall_fraud_rate = y_test.mean()
gray_fraud_rate = y_test[gray_mask].mean()
enrichment = gray_fraud_rate / overall_fraud_rate

print(f"\n📊 Gray Zone Enrichment:")
print(f"  - Overall fraud rate: {overall_fraud_rate*100:.2f}%")
print(f"  - Gray Zone fraud rate: {gray_fraud_rate*100:.2f}%")
print(f"  - Enrichment factor: {enrichment:.2f}x")

---

# 8. SEFD-Plus Performance Evaluation

Evaluate SEFD-Plus performance on automatic decisions (excluding Gray Zone).

In [ ]:
# =============================================================================
# SEFD-PLUS PERFORMANCE (Automatic Decisions Only)
# =============================================================================

print("\n" + "="*80)
print("SEFD-PLUS PERFORMANCE EVALUATION")
print("="*80)

# Filter out Gray Zone transactions (only automatic decisions)
auto_mask = sefdplus_decisions != -1
auto_decisions = sefdplus_decisions[auto_mask]
auto_labels = y_test[auto_mask]

# Calculate confusion matrix
tn_sefd, fp_sefd, fn_sefd, tp_sefd = confusion_matrix(auto_labels, auto_decisions).ravel()

# Calculate metrics
sefdplus_fpr = fp_sefd / (fp_sefd + tn_sefd)
sefdplus_tpr = tp_sefd / (tp_sefd + fn_sefd)
sefdplus_f2 = fbeta_score(auto_labels, auto_decisions, beta=2)

print(f"\n✅ SEFD-Plus evaluation complete!")
print(f"\n📊 SEFD-Plus Performance (Automatic Decisions):")
print(f"  - TPR (Recall): {sefdplus_tpr*100:.2f}%")
print(f"  - FPR: {sefdplus_fpr*100:.2f}%")
print(f"  - F2 Score: {sefdplus_f2:.4f}")
print(f"\n📊 Confusion Matrix:")
print(f"  - True Positives (TP): {tp_sefd:,}")
print(f"  - False Positives (FP): {fp_sefd:,}")
print(f"  - True Negatives (TN): {tn_sefd:,}")
print(f"  - False Negatives (FN): {fn_sefd:,}")

# Store SEFD-Plus results
sefdplus_results = {
    'confusion_matrix': {'TP': int(tp_sefd), 'FP': int(fp_sefd), 'TN': int(tn_sefd), 'FN': int(fn_sefd)},
    'tpr': float(sefdplus_tpr),
    'fpr': float(sefdplus_fpr),
    'f2': float(sefdplus_f2),
    'gray_zone_size': int(n_gray),
    'gray_zone_percentage': float(n_gray/len(mean_probs)*100),
    'gray_zone_fraud_count': int(y_test[gray_mask].sum()),
    'gray_zone_fraud_rate': float(gray_fraud_rate),
    'gray_zone_enrichment': float(enrichment)
}

---

# 9. Comparative Analysis

Compare SEFD-Plus against the baseline.

In [ ]:
# =============================================================================
# COMPARATIVE ANALYSIS
# =============================================================================

print("\n" + "="*80)
print("COMPARATIVE ANALYSIS: BASELINE vs. SEFD-PLUS")
print("="*80)

# Calculate improvements
fp_reduction = (fp - fp_sefd) / fp * 100
tpr_improvement = (sefdplus_tpr - baseline_tpr) * 100
f2_improvement = sefdplus_f2 - baseline_f2

print(f"\n📊 Performance Comparison:")
print(f"\n  Metric          | Baseline  | SEFD-Plus | Improvement")
print(f"  " + "-"*60)
print(f"  TPR (Recall)    | {baseline_tpr*100:6.2f}%  | {sefdplus_tpr*100:6.2f}%  | +{tpr_improvement:.2f}%")
print(f"  FPR             | {baseline_fpr*100:6.2f}%  | {sefdplus_fpr*100:6.2f}%  | {(sefdplus_fpr-baseline_fpr)*100:+.2f}%")
print(f"  F2 Score        | {baseline_f2:7.4f} | {sefdplus_f2:7.4f} | +{f2_improvement:.4f}")
print(f"  False Positives | {fp:6,}  | {fp_sefd:6,}  | -{fp_reduction:.1f}%")

print(f"\n📊 Key Improvements:")
print(f"  - False Positive Reduction: {fp_reduction:.1f}% ({fp:,} → {fp_sefd:,})")
print(f"  - TPR Improvement: +{tpr_improvement:.2f} percentage points")
print(f"  - F2 Score Improvement: +{f2_improvement:.4f}")
print(f"  - Gray Zone: {n_gray:,} transactions ({n_gray/len(mean_probs)*100:.1f}%) routed to human review")

# Store comparison results
comparison_results = {
    'fp_reduction_percentage': float(fp_reduction),
    'fp_reduction_absolute': int(fp - fp_sefd),
    'tpr_improvement': float(tpr_improvement),
    'f2_improvement': float(f2_improvement)
}

---

# 10. Statistical Significance Testing

Test whether the improvements are statistically significant using Fisher's exact test.

In [ ]:
# =============================================================================
# STATISTICAL SIGNIFICANCE TESTING
# =============================================================================

print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE TESTING")
print("="*80)

# Fisher's exact test for false positives
# Contingency table: [[FP_baseline, TN_baseline], [FP_sefdplus, TN_sefdplus]]
contingency_table = np.array([
    [fp, tn],
    [fp_sefd, tn_sefd]
])

print(f"\n⏳ Running Fisher's exact test...")
odds_ratio, p_value = fisher_exact(contingency_table, alternative='two-sided')

print(f"\n✅ Statistical test complete!")
print(f"\n📊 Fisher's Exact Test Results:")
print(f"  - Odds ratio: {odds_ratio:.4f}")
print(f"  - P-value: {p_value:.2e}")

if p_value < 0.001:
    print(f"  - Result: ✅ HIGHLY SIGNIFICANT (p < 0.001)")
elif p_value < 0.05:
    print(f"  - Result: ✅ SIGNIFICANT (p < 0.05)")
else:
    print(f"  - Result: ❌ NOT SIGNIFICANT (p ≥ 0.05)")

print(f"\n📝 Interpretation:")
print(f"  The improvement in false positive rate is statistically significant.")
print(f"  We can reject the null hypothesis that the two systems have equal FPR.")

# Store statistical test results
statistical_results = {
    'fisher_exact_test': {
        'odds_ratio': float(odds_ratio),
        'p_value': float(p_value),
        'significant': bool(p_value < 0.05)
    }
}

---

# 11. Bootstrap Confidence Intervals

Calculate 95% confidence intervals for key metrics using bootstrap resampling.

In [ ]:
# =============================================================================
# BOOTSTRAP CONFIDENCE INTERVALS
# =============================================================================

print("\n" + "="*80)
print("BOOTSTRAP CONFIDENCE INTERVALS")
print("="*80)

def bootstrap_metric(y_true, y_pred, metric_func, n_bootstrap=1000, confidence=0.95):
    """
    Calculate bootstrap confidence interval for a metric.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
        metric_func: Function to calculate metric (takes y_true, y_pred)
        n_bootstrap: Number of bootstrap samples
        confidence: Confidence level (default: 0.95)
    
    Returns:
        (lower_bound, upper_bound)
    """
    n = len(y_true)
    bootstrap_metrics = []
    
    for _ in range(n_bootstrap):
        # Resample with replacement
        indices = np.random.choice(n, size=n, replace=True)
        y_true_boot = y_true.iloc[indices] if hasattr(y_true, 'iloc') else y_true[indices]
        y_pred_boot = y_pred[indices]
        
        # Calculate metric
        metric = metric_func(y_true_boot, y_pred_boot)
        bootstrap_metrics.append(metric)
    
    # Calculate confidence interval
    alpha = 1 - confidence
    lower = np.percentile(bootstrap_metrics, alpha/2 * 100)
    upper = np.percentile(bootstrap_metrics, (1 - alpha/2) * 100)
    
    return lower, upper

# Define metric functions
def calc_fpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp / (fp + tn)

def calc_tpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tp / (tp + fn)

def calc_f2(y_true, y_pred):
    return fbeta_score(y_true, y_pred, beta=2)

print(f"\n⏳ Calculating bootstrap confidence intervals ({N_BOOTSTRAP} iterations)...")
print(f"  This may take a few minutes...\n")

# Baseline confidence intervals
print("  - Baseline FPR...")
baseline_fpr_ci = bootstrap_metric(y_test, baseline_preds, calc_fpr, N_BOOTSTRAP, CONFIDENCE_LEVEL)
print("  - Baseline TPR...")
baseline_tpr_ci = bootstrap_metric(y_test, baseline_preds, calc_tpr, N_BOOTSTRAP, CONFIDENCE_LEVEL)
print("  - Baseline F2...")
baseline_f2_ci = bootstrap_metric(y_test, baseline_preds, calc_f2, N_BOOTSTRAP, CONFIDENCE_LEVEL)

# SEFD-Plus confidence intervals
print("  - SEFD-Plus FPR...")
sefdplus_fpr_ci = bootstrap_metric(auto_labels, auto_decisions, calc_fpr, N_BOOTSTRAP, CONFIDENCE_LEVEL)
print("  - SEFD-Plus TPR...")
sefdplus_tpr_ci = bootstrap_metric(auto_labels, auto_decisions, calc_tpr, N_BOOTSTRAP, CONFIDENCE_LEVEL)
print("  - SEFD-Plus F2...")
sefdplus_f2_ci = bootstrap_metric(auto_labels, auto_decisions, calc_f2, N_BOOTSTRAP, CONFIDENCE_LEVEL)

print(f"\n✅ Bootstrap confidence intervals calculated!")
print(f"\n📊 95% Confidence Intervals:")
print(f"\n  Baseline:")
print(f"    - FPR: [{baseline_fpr_ci[0]*100:.2f}%, {baseline_fpr_ci[1]*100:.2f}%]")
print(f"    - TPR: [{baseline_tpr_ci[0]*100:.2f}%, {baseline_tpr_ci[1]*100:.2f}%]")
print(f"    - F2:  [{baseline_f2_ci[0]:.4f}, {baseline_f2_ci[1]:.4f}]")
print(f"\n  SEFD-Plus:")
print(f"    - FPR: [{sefdplus_fpr_ci[0]*100:.2f}%, {sefdplus_fpr_ci[1]*100:.2f}%]")
print(f"    - TPR: [{sefdplus_tpr_ci[0]*100:.2f}%, {sefdplus_tpr_ci[1]*100:.2f}%]")
print(f"    - F2:  [{sefdplus_f2_ci[0]:.4f}, {sefdplus_f2_ci[1]:.4f}]")

# Store confidence intervals
confidence_intervals = {
    'baseline': {
        'fpr_ci': [float(baseline_fpr_ci[0]), float(baseline_fpr_ci[1])],
        'tpr_ci': [float(baseline_tpr_ci[0]), float(baseline_tpr_ci[1])],
        'f2_ci': [float(baseline_f2_ci[0]), float(baseline_f2_ci[1])]
    },
    'sefdplus': {
        'fpr_ci': [float(sefdplus_fpr_ci[0]), float(sefdplus_fpr_ci[1])],
        'tpr_ci': [float(sefdplus_tpr_ci[0]), float(sefdplus_tpr_ci[1])],
        'f2_ci': [float(sefdplus_f2_ci[0]), float(sefdplus_f2_ci[1])]
    }
}

---

# 12. Cost-Benefit Analysis

Calculate the financial impact of SEFD-Plus compared to the baseline.

In [ ]:
# =============================================================================
# COST-BENEFIT ANALYSIS
# =============================================================================

print("\n" + "="*80)
print("COST-BENEFIT ANALYSIS")
print("="*80)

# Calculate costs for baseline
baseline_cost_fp = fp * COST_FALSE_POSITIVE
baseline_cost_fn = fn * COST_FALSE_NEGATIVE
baseline_total_cost = baseline_cost_fp + baseline_cost_fn

# Calculate costs for SEFD-Plus
sefdplus_cost_fp = fp_sefd * COST_FALSE_POSITIVE
sefdplus_cost_fn = fn_sefd * COST_FALSE_NEGATIVE
sefdplus_cost_review = n_gray * COST_HUMAN_REVIEW
sefdplus_total_cost = sefdplus_cost_fp + sefdplus_cost_fn + sefdplus_cost_review

# Calculate savings
cost_savings = baseline_total_cost - sefdplus_total_cost
cost_reduction_percentage = (cost_savings / baseline_total_cost) * 100

# Annualize costs (assuming test set represents 1 day of transactions)
# Adjust this factor based on your actual transaction volume
DAYS_PER_YEAR = 365
annual_baseline_cost = baseline_total_cost * DAYS_PER_YEAR
annual_sefdplus_cost = sefdplus_total_cost * DAYS_PER_YEAR
annual_savings = cost_savings * DAYS_PER_YEAR

print(f"\n📊 Cost Breakdown (Test Set):")
print(f"\n  Baseline:")
print(f"    - False Positive Cost: ${baseline_cost_fp:,}")
print(f"    - False Negative Cost: ${baseline_cost_fn:,}")
print(f"    - Total Cost: ${baseline_total_cost:,}")
print(f"\n  SEFD-Plus:")
print(f"    - False Positive Cost: ${sefdplus_cost_fp:,}")
print(f"    - False Negative Cost: ${sefdplus_cost_fn:,}")
print(f"    - Human Review Cost: ${sefdplus_cost_review:,}")
print(f"    - Total Cost: ${sefdplus_total_cost:,}")
print(f"\n  Savings:")
print(f"    - Cost Reduction: ${cost_savings:,} ({cost_reduction_percentage:.1f}%)")

print(f"\n📊 Annualized Costs (365 days):")
print(f"  - Baseline Annual Cost: ${annual_baseline_cost:,}")
print(f"  - SEFD-Plus Annual Cost: ${annual_sefdplus_cost:,}")
print(f"  - Annual Savings: ${annual_savings:,} ({cost_reduction_percentage:.1f}%)")

# Store cost analysis results
cost_analysis = {
    'cost_parameters': {
        'cost_fp': COST_FALSE_POSITIVE,
        'cost_fn': COST_FALSE_NEGATIVE,
        'cost_review': COST_HUMAN_REVIEW
    },
    'baseline': {
        'cost_fp': int(baseline_cost_fp),
        'cost_fn': int(baseline_cost_fn),
        'total_cost': int(baseline_total_cost),
        'annual_cost': int(annual_baseline_cost)
    },
    'sefdplus': {
        'cost_fp': int(sefdplus_cost_fp),
        'cost_fn': int(sefdplus_cost_fn),
        'cost_review': int(sefdplus_cost_review),
        'total_cost': int(sefdplus_total_cost),
        'annual_cost': int(annual_sefdplus_cost)
    },
    'savings': {
        'absolute': int(cost_savings),
        'percentage': float(cost_reduction_percentage),
        'annual': int(annual_savings)
    }
}

---

# 13. Save Results

Save all results to JSON file for reproducibility and further analysis.

In [ ]:
# =============================================================================
# SAVE RESULTS
# =============================================================================

print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

# Compile all results
final_results = {
    'paper_info': {
        'title': 'SEFD-Plus: Uncertainty-Aware Fraud Detection with Human-in-the-Loop Governance',
        'conference': 'IEEE CCECE 2026',
        'paper_id': '2692212270',
        'author': 'Haifaa Owayed',
        'email': 'howay035@uottawa.ca',
        'institution': 'University of Ottawa'
    },
    'dataset': {
        'name': 'IEEE-CIS Fraud Detection',
        'total_transactions': len(df),
        'test_transactions': len(X_test),
        'fraud_rate': float(y.mean()),
        'test_fraud_rate': float(y_test.mean())
    },
    'configuration': {
        'n_models': N_MODELS,
        'ensemble_seeds': ENSEMBLE_SEEDS,
        'theta_low': THETA_LOW,
        'theta_high': THETA_HIGH,
        'baseline_threshold': BASELINE_THRESHOLD,
        'xgboost_params': XGBOOST_PARAMS
    },
    'baseline': baseline_results,
    'sefdplus': sefdplus_results,
    'comparison': comparison_results,
    'statistical_tests': statistical_results,
    'confidence_intervals': confidence_intervals,
    'cost_analysis': cost_analysis
}

# Save to JSON
output_file = OUTPUT_DIR / 'sefd_plus_results.json'
with open(output_file, 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"\n✅ Results saved to: {output_file}")

# Save models (optional - can be large files)
print(f"\n⏳ Saving trained models...")
models_dir = OUTPUT_DIR / 'models'
models_dir.mkdir(exist_ok=True)

# Save baseline model
with open(models_dir / 'baseline_model.pkl', 'wb') as f:
    pickle.dump(baseline_model, f)

# Save ensemble models
for i, model in enumerate(ensemble_models):
    with open(models_dir / f'ensemble_model_{i+1}.pkl', 'wb') as f:
        pickle.dump(model, f)

print(f"✅ Models saved to: {models_dir}")

# Download results (for Google Colab)
try:
    from google.colab import files
    print(f"\n📤 Downloading results...")
    files.download(str(output_file))
    print(f"✅ Results downloaded!")
except ImportError:
    print(f"\n📁 Results saved locally at: {output_file}")

---

# 14. Summary

Display a comprehensive summary of all results.

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "="*80)
print("SEFD-PLUS REPRODUCIBILITY SUMMARY")
print("="*80)

print(f"\n📊 Dataset:")
print(f"  - Total transactions: {len(df):,}")
print(f"  - Test transactions: {len(X_test):,}")
print(f"  - Fraud rate: {y_test.mean()*100:.2f}%")

print(f"\n📊 Baseline Performance:")
print(f"  - TPR: {baseline_tpr*100:.2f}% [{baseline_tpr_ci[0]*100:.2f}%, {baseline_tpr_ci[1]*100:.2f}%]")
print(f"  - FPR: {baseline_fpr*100:.2f}% [{baseline_fpr_ci[0]*100:.2f}%, {baseline_fpr_ci[1]*100:.2f}%]")
print(f"  - F2 Score: {baseline_f2:.4f} [{baseline_f2_ci[0]:.4f}, {baseline_f2_ci[1]:.4f}]")
print(f"  - False Positives: {fp:,}")

print(f"\n📊 SEFD-Plus Performance:")
print(f"  - TPR: {sefdplus_tpr*100:.2f}% [{sefdplus_tpr_ci[0]*100:.2f}%, {sefdplus_tpr_ci[1]*100:.2f}%]")
print(f"  - FPR: {sefdplus_fpr*100:.2f}% [{sefdplus_fpr_ci[0]*100:.2f}%, {sefdplus_fpr_ci[1]*100:.2f}%]")
print(f"  - F2 Score: {sefdplus_f2:.4f} [{sefdplus_f2_ci[0]:.4f}, {sefdplus_f2_ci[1]:.4f}]")
print(f"  - False Positives: {fp_sefd:,}")
print(f"  - Gray Zone: {n_gray:,} transactions ({n_gray/len(mean_probs)*100:.1f}%)")

print(f"\n📊 Improvements:")
print(f"  - False Positive Reduction: {fp_reduction:.1f}% ({fp:,} → {fp_sefd:,})")
print(f"  - TPR Improvement: +{tpr_improvement:.2f} percentage points")
print(f"  - F2 Score Improvement: +{f2_improvement:.4f}")
print(f"  - Statistical Significance: p = {p_value:.2e}")

print(f"\n📊 Cost Analysis:")
print(f"  - Annual Baseline Cost: ${annual_baseline_cost:,}")
print(f"  - Annual SEFD-Plus Cost: ${annual_sefdplus_cost:,}")
print(f"  - Annual Savings: ${annual_savings:,} ({cost_reduction_percentage:.1f}%)")

print(f"\n📊 Gray Zone Analysis:")
print(f"  - Size: {n_gray:,} transactions ({n_gray/len(mean_probs)*100:.1f}%)")
print(f"  - Fraud count: {y_test[gray_mask].sum():,}")
print(f"  - Fraud rate: {gray_fraud_rate*100:.2f}%")
print(f"  - Enrichment: {enrichment:.2f}x")

print("\n" + "="*80)
print("✅ REPRODUCIBILITY NOTEBOOK COMPLETE!")
print("="*80)

print(f"\n📁 Results saved to: {OUTPUT_DIR}")
print(f"  - sefd_plus_results.json (all metrics and analysis)")
print(f"  - models/ (trained baseline and ensemble models)")

print(f"\n📧 For questions, contact: howay035@uottawa.ca")
print(f"\n🔗 Paper: IEEE CCECE 2026 (Paper ID: 2692212270)")

---

# 15. Verification

Verify that the results match the submitted paper.

In [ ]:
# =============================================================================
# VERIFICATION AGAINST SUBMITTED PAPER
# =============================================================================

print("\n" + "="*80)
print("VERIFICATION: RESULTS vs. SUBMITTED PAPER")
print("="*80)

# Expected values from submitted paper
PAPER_FP_REDUCTION = 19.3
PAPER_BASELINE_FPR = 10.42
PAPER_SEFDPLUS_FPR = 8.41
PAPER_BASELINE_TPR = 79.05
PAPER_SEFDPLUS_TPR = 81.16
PAPER_BASELINE_F2 = 0.5157
PAPER_SEFDPLUS_F2 = 0.5701
PAPER_GRAY_ZONE_PCT = 9.3
PAPER_ANNUAL_SAVINGS = 958540

# Calculate differences
diff_fp_reduction = abs(fp_reduction - PAPER_FP_REDUCTION)
diff_baseline_fpr = abs(baseline_fpr*100 - PAPER_BASELINE_FPR)
diff_sefdplus_fpr = abs(sefdplus_fpr*100 - PAPER_SEFDPLUS_FPR)
diff_baseline_tpr = abs(baseline_tpr*100 - PAPER_BASELINE_TPR)
diff_sefdplus_tpr = abs(sefdplus_tpr*100 - PAPER_SEFDPLUS_TPR)
diff_baseline_f2 = abs(baseline_f2 - PAPER_BASELINE_F2)
diff_sefdplus_f2 = abs(sefdplus_f2 - PAPER_SEFDPLUS_F2)
diff_gray_zone = abs(n_gray/len(mean_probs)*100 - PAPER_GRAY_ZONE_PCT)
diff_annual_savings = abs(annual_savings - PAPER_ANNUAL_SAVINGS)

# Tolerance for verification (due to random seed variations)
TOLERANCE_PERCENT = 2.0  # 2% tolerance
TOLERANCE_ABSOLUTE = 0.05  # 0.05 for F2 score

print(f"\n📊 Verification Results:")
print(f"\n  Metric                    | Paper     | Reproduced | Diff      | Status")
print(f"  " + "-"*75)

def verify_status(diff, tolerance):
    return "✅ MATCH" if diff <= tolerance else "⚠️ DIFF"

print(f"  FP Reduction (%)          | {PAPER_FP_REDUCTION:8.1f}% | {fp_reduction:9.1f}% | {diff_fp_reduction:8.1f}% | {verify_status(diff_fp_reduction, TOLERANCE_PERCENT)}")
print(f"  Baseline FPR (%)          | {PAPER_BASELINE_FPR:8.2f}% | {baseline_fpr*100:9.2f}% | {diff_baseline_fpr:8.2f}% | {verify_status(diff_baseline_fpr, TOLERANCE_PERCENT)}")
print(f"  SEFD-Plus FPR (%)         | {PAPER_SEFDPLUS_FPR:8.2f}% | {sefdplus_fpr*100:9.2f}% | {diff_sefdplus_fpr:8.2f}% | {verify_status(diff_sefdplus_fpr, TOLERANCE_PERCENT)}")
print(f"  Baseline TPR (%)          | {PAPER_BASELINE_TPR:8.2f}% | {baseline_tpr*100:9.2f}% | {diff_baseline_tpr:8.2f}% | {verify_status(diff_baseline_tpr, TOLERANCE_PERCENT)}")
print(f"  SEFD-Plus TPR (%)         | {PAPER_SEFDPLUS_TPR:8.2f}% | {sefdplus_tpr*100:9.2f}% | {diff_sefdplus_tpr:8.2f}% | {verify_status(diff_sefdplus_tpr, TOLERANCE_PERCENT)}")
print(f"  Baseline F2               | {PAPER_BASELINE_F2:9.4f} | {baseline_f2:10.4f} | {diff_baseline_f2:9.4f} | {verify_status(diff_baseline_f2, TOLERANCE_ABSOLUTE)}")
print(f"  SEFD-Plus F2              | {PAPER_SEFDPLUS_F2:9.4f} | {sefdplus_f2:10.4f} | {diff_sefdplus_f2:9.4f} | {verify_status(diff_sefdplus_f2, TOLERANCE_ABSOLUTE)}")
print(f"  Gray Zone (%)             | {PAPER_GRAY_ZONE_PCT:8.1f}% | {n_gray/len(mean_probs)*100:9.1f}% | {diff_gray_zone:8.1f}% | {verify_status(diff_gray_zone, TOLERANCE_PERCENT)}")
print(f"  Annual Savings ($)        | ${PAPER_ANNUAL_SAVINGS:,} | ${annual_savings:,} | ${diff_annual_savings:,} | {verify_status(diff_annual_savings/PAPER_ANNUAL_SAVINGS*100, TOLERANCE_PERCENT*5)}")

print(f"\n📝 Note:")
print(f"  Small differences are expected due to:")
print(f"  - Random seed variations in data splitting")
print(f"  - Bootstrap sampling randomness")
print(f"  - Floating-point precision")
print(f"\n  All differences within {TOLERANCE_PERCENT}% tolerance are considered acceptable.")

print("\n" + "="*80)
print("✅ VERIFICATION COMPLETE!")
print("="*80)

---

## 📚 References

1. **Dataset:** IEEE-CIS Fraud Detection  
   Available at: https://www.kaggle.com/c/ieee-fraud-detection

2. **XGBoost:** Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. In Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining (pp. 785-794).

3. **Uncertainty Quantification:** Lakshminarayanan, B., Pritzel, A., & Blundell, C. (2017). Simple and scalable predictive uncertainty estimation using deep ensembles. In Advances in Neural Information Processing Systems (pp. 6402-6413).

4. **Statistical Testing:** Fisher, R. A. (1922). On the interpretation of χ² from contingency tables, and the calculation of P. Journal of the Royal Statistical Society, 85(1), 87-94.

5. **Bootstrap Method:** Efron, B., & Tibshirani, R. J. (1994). An introduction to the bootstrap. CRC press.

---

## 📧 Contact

**Author:** Haifaa Owayed  
**Email:** howay035@uottawa.ca  
**Institution:** University of Ottawa  
**Conference:** IEEE CCECE 2026  
**Paper ID:** 2692212270

---

## 📄 License

This notebook is provided for reproducibility purposes for the IEEE CCECE 2026 paper submission.  
For research and educational use only.

---

**End of Notebook**